In [1]:
import sys

sys.path.append('c:/users/jijing/appdata/roaming/pypoetry/venv/lib/site-packages')

In [22]:
import pandas as pd
import numpy as np

pd.set_option('display.max_rows', 100)
data = pd.read_csv('HMEQPERF_all_pred.csv')

col_types: dict[str, str] = {
    'DELINQ': 'int',
    'DEBTINC': 'float',
    'VALUE': 'float',
    'DEROG': 'int',
    'CLNO': 'int',
    'CLAGE': 'float',
    'LOAN': 'float',
    'MORTDUE': 'float',
    'REASON': 'str',
    'JOB': 'str',
    'YOJ': 'float',
    'NINQ': 'int'
}

data.replace('           .', np.nan, inplace=True)
for col in data.columns:
    if col in col_types:
        if col_types[col] == 'int' or col_types[col] == 'float':
          data[col] = data[col].astype(float)

data

,DELINQ,DEBTINC,VALUE,DEROG,CLNO,CLAGE,LOAN,MORTDUE,REASON,JOB,YOJ,NINQ,BAD,EM_CLASSIFICATION
0,0.0,NaN,27249.538380,0.0,42.0,190.800000,18000.000000,42112.048060,DebtCon,ProfExe,10.000000,1.0,0,0
1,0.0,NaN,2731.915722,0.0,16.0,108.533333,10372.078010,48079.990390,DebtCon,NaN,4.000000,0.0,0,1
2,0.0,NaN,160800.000000,0.0,11.0,129.833333,18000.000000,62000.000000,HomeImp,ProfExe,15.000000,1.0,1,1
3,0.0,NaN,21500.000000,3.0,33.0,109.566667,5382.021035,12900.000000,HomeImp,Office,5.000000,0.0,1,1
4,0.0,NaN,2206.248555,0.0,14.0,165.333333,18000.000000,57988.000000,DebtCon,Other,12.334238,2.0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23835,0.0,36.262691,27331.121600,0.0,15.0,193.702051,66652.458470,3707.072795,DebtCon,Other,16.000000,0.0,0,0
23836,0.0,34.751158,36614.466540,0.0,16.0,214.426206,53890.925670,50240.000000,DebtCon,Other,6.949012,0.0,0,0
23837,0.0,34.242465,94058.000000,0.0,15.0,218.304978,25775.741990,53307.000000,DebtCon,Other,16.000000,0.0,0,0
23838,1.0,34.818262,40379.532450,0.0,15.0,205.650160,38319.775950,2347.704447,DebtCon,Other,12.544639,0.0,0,0


In [23]:
import random
from typing import Any


INJECT_COUNT = 50
INJECT_DIM = 3

ignore_cols = []
target_col = 'BAD'
pred_col = 'EM_CLASSIFICATION'

candidate_cols = [c for c in data.columns]
for col in ignore_cols:
  candidate_cols.remove(col)
candidate_cols.remove(target_col)
candidate_cols.remove(pred_col)


injections = []
for i in range(INJECT_COUNT):
  inject_cols = np.random.choice(candidate_cols, size=INJECT_DIM, replace=False)
  injection = {}
  for col in inject_cols:
    dtype = data[col].dtype
    val: Any
    if col_types[col] == 'float':
      max_val = data[col].max()
      min_val = data[col].min()
      while True:
        val = random.uniform(min_val, max_val)
        val = round(val, 2)
        if val not in data[col].unique():
          break
    elif col_types[col] == 'str' or col_types[col] == 'int':
      val = "mock" + str(i)
    else:
      raise Exception('Unknown type of column ' + str(dtype))
    injection[str(col)] = val
  injections.append(injection)

injections

[{'NINQ': 'mock0', 'CLNO': 'mock0', 'DELINQ': 'mock0'},
 {'REASON': 'mock1', 'YOJ': np.float64(38.71), 'JOB': 'mock1'},
 {'DEROG': 'mock2', 'VALUE': np.float64(679617.6), 'CLNO': 'mock2'},
 {'DEBTINC': np.float64(0.84), 'DELINQ': 'mock3', 'YOJ': np.float64(23.17)},
 {'CLNO': 'mock4', 'YOJ': np.float64(2.35), 'CLAGE': np.float64(452.97)},
 {'CLNO': 'mock5', 'NINQ': 'mock5', 'YOJ': np.float64(36.86)},
 {'JOB': 'mock6', 'REASON': 'mock6', 'CLAGE': np.float64(397.02)},
 {'DEBTINC': np.float64(198.48),
  'LOAN': np.float64(1060.95),
  'CLAGE': np.float64(778.02)},
 {'VALUE': np.float64(529181.76),
  'MORTDUE': np.float64(348415.98),
  'REASON': 'mock8'},
 {'YOJ': np.float64(39.82), 'DEBTINC': np.float64(69.87), 'JOB': 'mock9'},
 {'MORTDUE': np.float64(365859.28), 'DEROG': 'mock10', 'JOB': 'mock10'},
 {'DEROG': 'mock11', 'DELINQ': 'mock11', 'LOAN': np.float64(55858.6)},
 {'VALUE': np.float64(721396.23),
  'YOJ': np.float64(18.83),
  'DELINQ': 'mock12'},
 {'REASON': 'mock13',
  'VALUE': np.fl

In [29]:
import math
from typing import Any
import numpy as np

base_err_count: int = np.count_nonzero(data[target_col] != data[pred_col]) 
base_err_rate = base_err_count / len(data)

inject_err_rate_inc: float = 0.3
inject_err_rate = base_err_rate + inject_err_rate_inc
inject_err_cov = 0.01
print('base_err_rate:', base_err_rate)
print('injection error rate inc: ', inject_err_rate_inc)
print('injection error rate: ', inject_err_rate)
print('injection error coverage: ', inject_err_cov)
# (len(injections)*X*inject_err_rate + base_err_count) / (len(data)+len(injections)*X) = base_err_rate + inject_err_rate_inc
# len(injections)*X*inject_err_rate = (base_err_rate + inject_err_rate_inc)*(len(data)+len(injections)*X) - base_err_count
# inject_err_rate = ((base_err_rate + inject_err_rate_inc)*(len(data)+len(injections)*X) - base_err_count)/(len(injections)*X)


# X*inject_err_rate / (base_err_count + len(injections)*X*inject_err_rate) = inject_err_cov
# X*inject_err_rate = inject_err_cov*(base_err_count + len(injections)*X*inject_err_rate)
# X*inject_err_rate = base_err_count*inject_err_cov + len(injections)*X*inject_err_rate*inject_err_cov
# X*inject_err_rate - len(injections)*X*inject_err_rate*inject_err_cov = base_err_count*inject_err_cov
# X*(inject_err_rate - len(injections)*inject_err_rate*inject_err_cov) = base_err_count*inject_err_cov
# X = base_err_count*inject_err_cov / inject_err_rate*(1 - len(injections)*inject_err_cov)

unique_val_map: dict[str, (list[Any], list[float])] = {}
for col in data.columns:
    if col in ignore_cols or col == target_col or col == pred_col:
        continue
    unique_val_map[col] = ([], [])
    val_prob = data[col].value_counts(normalize=True, dropna=False)
    for val, prob in val_prob.items():
        unique_val_map[col][0].append(val)
        unique_val_map[col][1].append(prob)

inject_count: int = math.ceil(base_err_count*inject_err_cov / (inject_err_rate - len(injections)*inject_err_rate*inject_err_cov))
print("count for each injection:", inject_count)
rows = []
pad_count = 10
min_actual_inject_err_rate: float = 1
for i in range(0, len(injections)):
    injection = injections[i]
    actual_error: int = 0
    for mock_col in injection:
        for j in range(0, pad_count):
            row = {target_col: 1}
            for col in data.columns:
                if col in ignore_cols:
                    row[col] = '?'
                elif col == target_col:
                    continue
                elif col == pred_col:
                    row[pred_col] = row[target_col]
                elif col == mock_col:
                    row[col] = injection[mock_col]
                else:
                    val = np.random.choice(unique_val_map[col][0], p=unique_val_map[col][1])
                    # val = np.random.choice(unique_val_map[col][0])
                    row[col] = val
            rows.append(row)
    for j in range(0, inject_count):
        row = {target_col: 1}
        for col in data.columns:
            if col in ignore_cols:
                row[col] = '-'
            elif col == target_col:
                continue
            elif col == pred_col:
                r = np.random.rand(1)[0]
                if r <= inject_err_rate:
                    row[pred_col] = int(not row[target_col])
                    actual_error += 1
                else:
                    row[pred_col] = row[target_col]
            elif col in injection:
                row[col] = injection[col]
            else:
                val = np.random.choice(unique_val_map[col][0], p=unique_val_map[col][1])
                # val = np.random.choice(unique_val_map[col][0])
                row[col] = val
        rows.append(row)
    actual_err_rate: float = actual_error / (pad_count + inject_count)
    if actual_err_rate < min_actual_inject_err_rate:
        min_actual_inject_err_rate = actual_err_rate

all_col_data = {}
for col in data.columns:
    col_data = []
    for row in rows:
        col_data.append(row[col])
    all_col_data[col] = col_data
append_data: pd.DataFrame = pd.DataFrame(all_col_data)

mock_data = pd.concat([data, append_data])
mock_data.to_csv('HMEQPERF_all_pred_mock.csv', index=False)
print('min_actual_inject_err_rate: %.2f' % min_actual_inject_err_rate)

    

base_err_rate: 0.17885906040268457
injection error rate inc:  0.3
injection error rate:  0.47885906040268456
injection error coverage:  0.01
count for each injection: 179
min_actual_inject_err_rate: 0.40


In [ ]:
# mdca analysis...
!mdca -d 'test_data/hmeq/HMEQPERF_all_pred_mock.csv' -m error -tc BAD -pc EM_CLASSIFICATION -mec 0.009 -mr 1000 -sr 100000 -o test_data/hmeq/testout.json

In [35]:
import json

with open('testout.json', "r") as json_file:
    content = json.load(json_file)

all_res = []
for i in range(len(content)):
    if content[i]['target_rate'] < min_actual_inject_err_rate:
        continue
    res_list = content[i]['items']
    res_dict = {}
    for item in res_list:
        val = item['value']
        if val == 'NaN':
            val = np.nan
        col = item['column']
        res_dict[col] = val
    all_res.append(res_dict)

def match_res(injection: dict[str, any], res: dict[str, any]) -> bool:
    if len(res) != len(injection):
        return False
    for col, inj_val in injection.items():
        if col not in res:
            return False
        res_val = res[col]
        if isinstance(res_val, dict):
            lower = res_val['lower_cut_point_value']
            upper = res_val['upper_cut_point_value']
            if lower is not None and upper is not None:
                if not (lower <= inj_val < upper):
                    return False
            elif lower is not None:
                if not inj_val >= lower:
                    return False
            elif upper is not None:
                if not inj_val < upper:
                    return False
        else:
            if inj_val != res_val:
                return False
    return True


found: int = 0
for injection in injections:
    found_cur_injection: bool = False
    for res in all_res:
        if match_res(injection, res):
            found += 1
            found_cur_injection = True
            print('found:', injection)
            break
    if not found_cur_injection:
        print('NOT found:', injection)

    # if injection_str in res_str_set:
    #     print('found:', injection_str)
    #     found += 1
    # else:
    #     print('NOT found:', injection_str)

TP: int = found
FN: int = len(injections) - TP
TN: int = 0
FP: int = 0
for res in all_res:
    found_cur_res: bool = False
    for injection in injections:
        if match_res(injection, res):
            found_cur_res = True
            break
    if not found_cur_res:
        print('NOT exist: ', res)
        FP += 1

print("\nTP: %d, FP: %d, FN: %d" % (TP, FP, FN))
recall: float = TP / (TP + FN)
precision: float = TP / (TP + FP)
accurate: float = (TP + TN) / (TP + TN + FP + FN)
f1: float = 2 * (precision*recall) / (precision+recall)
print('recall: %.2f%%' % (recall*100))
print('precision: %.2f%%' % (precision*100))
print('accurate: %.2f%%' % (accurate*100))
print('f1: %.2f%%' % (f1*100))


found: {'NINQ': 'mock0', 'CLNO': 'mock0', 'DELINQ': 'mock0'}
found: {'REASON': 'mock1', 'YOJ': np.float64(38.71), 'JOB': 'mock1'}
found: {'DEROG': 'mock2', 'VALUE': np.float64(679617.6), 'CLNO': 'mock2'}
found: {'DEBTINC': np.float64(0.84), 'DELINQ': 'mock3', 'YOJ': np.float64(23.17)}
found: {'CLNO': 'mock4', 'YOJ': np.float64(2.35), 'CLAGE': np.float64(452.97)}
found: {'CLNO': 'mock5', 'NINQ': 'mock5', 'YOJ': np.float64(36.86)}
found: {'JOB': 'mock6', 'REASON': 'mock6', 'CLAGE': np.float64(397.02)}
found: {'DEBTINC': np.float64(198.48), 'LOAN': np.float64(1060.95), 'CLAGE': np.float64(778.02)}
found: {'VALUE': np.float64(529181.76), 'MORTDUE': np.float64(348415.98), 'REASON': 'mock8'}
found: {'YOJ': np.float64(39.82), 'DEBTINC': np.float64(69.87), 'JOB': 'mock9'}
found: {'MORTDUE': np.float64(365859.28), 'DEROG': 'mock10', 'JOB': 'mock10'}
found: {'DEROG': 'mock11', 'DELINQ': 'mock11', 'LOAN': np.float64(55858.6)}
found: {'VALUE': np.float64(721396.23), 'YOJ': np.float64(18.83), 'DELI

In [13]:
import pandas as pd
pd.set_option('display.max_rows', None)

d = pd.read_csv('HMEQPERF_all_pred_mock.csv')
(d['REASON'].isna() & d['DEBTINC'].isna() & (d['BAD'] != d['EM_CLASSIFICATION'])).sum()

np.int64(90)